# 02 — Validation

- **V1** duplicates and leaks → remove before split
- **V2** is `is_valid` like train → use as holdout
- **V3** score noise on holdout and sample → which differences to trust
- **V4** first CV ↔ LB point → which score to trust
- **V5** save split → same for all models

### Imports and data

In [1]:
import ast
import json
import os
import re
import string
from collections import Counter
from typing import List

import numpy as np
import pandas as pd

DATA = '../data'

uk = pd.read_csv(f'{DATA}/uk_geo_dataset.csv')
sample = pd.read_csv(f'{DATA}/labeling_sample.csv')
test = pd.read_csv(f'{DATA}/test.csv')

uk['spans'] = uk['loc_markers'].map(ast.literal_eval)
sample['locations'] = sample['locations'].map(ast.literal_eval)

### Competition metric (Kaggle, unchanged)

In [2]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p, = 0.0, 0.0, 0.0, 0.0
    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = set([hard_processing(s) for s in y_true_sample])
        y_pred_sample = set([hard_processing(s) for s in y_pred_sample])
        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 8)

## V1. Duplicates and leaks

In [3]:
valid_texts = uk.loc[uk['is_valid'] == 1, 'text']
train_texts = uk.loc[uk['is_valid'] == 0, 'text']

print('Duplicate texts:', uk['text'].duplicated().sum())
print('is_valid texts in train:', valid_texts.isin(train_texts).sum())
print('Sample texts in train:', sample['text'].isin(uk['text']).sum())
print('Test texts in train:', test['text'].isin(uk['text']).sum())

Duplicate texts: 2454


is_valid texts in train: 52


Sample texts in train: 0


Test texts in train: 0


In [4]:
uk = uk.sort_values('is_valid').drop_duplicates('text', keep='first').sort_index()
print('After dedup:', len(uk))

After dedup: 1007546


## V2. Is `is_valid` like train

In [5]:
uk['n_loc'] = uk['spans'].map(len)
uk['has_loc'] = uk['n_loc'] > 0
uk['n_chars'] = uk['text'].str.len()

uk.groupby('is_valid')[['has_loc', 'n_loc', 'n_chars']].mean().round(3)

,has_loc,n_loc,n_chars
is_valid,,,
0,0.231,0.342,102.101
1,0.231,0.331,102.734


In [6]:
train = uk[uk['is_valid'] == 0]
holdout = uk[uk['is_valid'] == 1]
print('train:', len(train), '| holdout:', len(holdout), '| holdout locations:', holdout['n_loc'].sum())

train: 997598 | holdout: 9948 | holdout locations: 3293


## V3. Score noise: dictionary probe
Probe: words and bigrams labeled as locations ≥ 5 times in train (capitalized, ≥ 3 chars). Lexicon from train only.

In [7]:
WORD_RE = re.compile(r"\w+(?:['’ʼ]\w+)*")

counts = Counter(t[s:e] for t, spans in zip(train['text'], train['spans']) for s, e in spans)
lexicon = {loc for loc, n in counts.items() if n >= 5 and loc[0].isupper() and len(loc) >= 3}
print('Lexicon size:', len(lexicon))


def predict_dictionary(text):
    words = WORD_RE.findall(text)
    candidates = words + [a + ' ' + b for a, b in zip(words, words[1:])]
    return [c for c in candidates if c in lexicon]


predict_dictionary('Сьогодні з Києва до Хелма вперше вирушив поїзд')

Lexicon size: 4563


['Києва']

In [8]:
def bootstrap_ci(y_true, y_pred, n=500):
    rng = np.random.default_rng(42)
    scores = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        scores.append(comp_metric([y_true[i] for i in idx], [y_pred[i] for i in idx]))
    return np.percentile(scores, [2.5, 97.5])


holdout_true = [[t[s:e] for s, e in spans] for t, spans in zip(holdout['text'], holdout['spans'])]
holdout_pred = [predict_dictionary(t) for t in holdout['text']]
sample_true = list(sample['locations'])
sample_pred = [predict_dictionary(t) for t in sample['text']]

for name, y_true, y_pred in [('holdout', holdout_true, holdout_pred), ('sample', sample_true, sample_pred)]:
    low, high = bootstrap_ci(y_true, y_pred)
    print(f'{name:8} F1 = {comp_metric(y_true, y_pred):.3f}   95% CI = [{low:.3f}, {high:.3f}]')

holdout  F1 = 0.675   95% CI = [0.661, 0.688]
sample   F1 = 0.405   95% CI = [0.162, 0.586]


In [9]:
for true, pred in zip(sample_true, sample_pred):
    if true or pred:
        print('gold:', true, '| probe:', pred)

gold: ['Проспект Бандери'] | probe: ['Бандери', 'Оболонь']
gold: ['Київщині', 'Чорнобиль', 'Київ', 'Тетерів', 'Фастів', 'Біла Церква', 'Миронівка', 'Бориспіль', 'Яготин', 'Баришівка'] | probe: ['Київщині', 'Чорнобиль', 'Київ', 'Фастів', 'Бориспіль', 'YouTube', 'Біла Церква']
gold: ['проспекті Івасюка'] | probe: ['Оболонь']
gold: ['Києва', 'Хелма'] | probe: ['Києва', 'Укрзалізниці']
gold: ['Києву'] | probe: ['Києва', 'Сотні', 'Києву', 'України', 'Україні']
gold: [] | probe: ['України']
gold: ['ТЦ Аркадія'] | probe: []
gold: [] | probe: ['Київський']
gold: ['Фастовском', 'Бучанском'] | probe: []
gold: ['Троєщині'] | probe: ['Троєщині', 'Київ']
gold: ['Шевченківському районі'] | probe: ['Шевченківському', 'Києва', 'Шевченківському районі']
gold: ['Героїв Дніпра'] | probe: ['Дніпра', 'Оболонь']
gold: [] | probe: ['Київському', 'Шевченка', 'Тараса Шевченка']
gold: ["вул. Солом'янська, 28"] | probe: []
gold: [] | probe: ['Президента']
gold: [] | probe: ['Місяць']
gold: ['Києвом', 'Васильково

## V4. Probe submission

In [10]:
os.makedirs(f'{DATA}/submissions', exist_ok=True)
submission = pd.DataFrame({
    'text_id': test['text_id'],
    'locations': [json.dumps(predict_dictionary(t), ensure_ascii=False) for t in test['text']],
})
submission.to_csv(f'{DATA}/submissions/probe_dictionary.csv', index=False)
submission.head()

,text_id,locations
0,0,"[""Київ""]"
1,1,"[""Києві"", ""Києва"", ""Черновецького"", ""Шулявсько..."
2,2,"[""Полтавської"", ""Президента"", ""Полтавської обл..."
3,3,[]
4,4,[]


In [11]:
PUBLIC_LB = 0.41896

print('holdout:', round(comp_metric(holdout_true, holdout_pred), 3))
print('sample: ', round(comp_metric(sample_true, sample_pred), 3))
print('LB:     ', PUBLIC_LB)

holdout: 0.675
sample:  0.405
LB:      0.41896


## V5. Save split

In [12]:
os.makedirs(f'{DATA}/splits', exist_ok=True)
split = pd.DataFrame({'row_id': uk.index, 'split': np.where(uk['is_valid'] == 1, 'holdout', 'train')})
split.to_csv(f'{DATA}/splits/split.csv', index=False)
split['split'].value_counts()

split
train      997598
holdout      9948
Name: count, dtype: int64

## Summary

| # | Result | Decision |
|---|---|---|
| V1 | 2,454 duplicates; 52 `is_valid` texts also in train; no sample/test overlap | Deduplicated, no holdout leak |
| V2 | `is_valid` matches train: 23.1% texts with LOC, ~102 chars | Holdout = `is_valid`: 9,948 sentences, 3,293 locations |
| V3 | Probe: holdout 0.675 [0.661, 0.688], sample 0.405 [0.162, 0.586] | Trust holdout differences > ~1.5 pt, sample > ~20 pt |
| V4 | LB 0.419 ≈ sample (0.405); holdout overestimates by ~0.26 | Holdout ranks models, does not predict LB |
| V5 | `data/splits/split.csv` | One split for all models |

Holdout → sample gap (0.68 → 0.41) is domain shift: the probe fails where context matters
(channel signatures, «України» inside organization names, unseen streets).